---

## Key Takeaways - Data Fetching

✅ **What we accomplished:**
1. Fetched Vermont town boundaries from ArcGIS REST endpoint
2. Implemented smart caching to avoid redundant downloads
3. Loaded GeoJSON data into a GeoPandas GeoDataFrame
4. Explored the data structure, attributes, and coordinate system
5. Identified the town name field for future selection

💡 **Key Concepts:**
- **REST APIs** provide programmatic access to GIS data
- **Caching** improves performance and enables offline work
- **GeoDataFrame** extends pandas with spatial capabilities
- **CRS (Coordinate Reference System)** defines how coordinates map to real-world locations

🔜 **Next Steps:**
In the next section, we'll create an interactive dropdown to select towns and view their properties.

---

In [ ]:
# Identify the town name column
# Common field names: TOWNNAME, TOWN, NAME, etc.
name_candidates = ['TOWNNAME', 'TOWN', 'NAME', 'Town', 'name']
town_name_field = None

for candidate in name_candidates:
    if candidate in towns_gdf.columns:
        town_name_field = candidate
        break

if town_name_field is None:
    # If no standard field found, show available fields
    print("⚠️  Could not automatically detect town name field.")
    print("Available fields:", list(towns_gdf.columns))
    # Assume first non-geometry column
    town_name_field = [c for c in towns_gdf.columns if c != 'geometry'][0]
    print(f"Using field: {town_name_field}")

# Get sorted list of town names
town_names = sorted(towns_gdf[town_name_field].unique())

print(f"\n✓ Found {len(town_names)} Vermont towns")
print(f"\nTown name field: '{town_name_field}'")
print(f"\nSample towns (first 10):")
for name in town_names[:10]:
    print(f"  • {name}")
print(f"  ...")
print(f"  • {town_names[-1]}")

### Town Name List

Let's get a sorted list of all Vermont town names. We'll use this in the next step to create our interactive selector.

In [ ]:
# Display first few towns
# Drop geometry column for cleaner display (it's very long)
display_cols = [col for col in towns_gdf.columns if col != 'geometry']

print("\nFirst 5 Towns:")
print("=" * 60)
towns_gdf[display_cols].head()

### Sample Town Data

Let's look at a few example towns to understand the data structure better.

In [ ]:
# Display column names and types
print("Available Columns:")
print("=" * 60)
for col in towns_gdf.columns:
    dtype = towns_gdf[col].dtype
    if col != 'geometry':
        sample = towns_gdf[col].iloc[0] if len(towns_gdf) > 0 else None
        print(f"  {col:20s} ({dtype}) - Example: {sample}")
    else:
        print(f"  {col:20s} ({dtype})")

### Examining Attributes

Let's see what attribute fields are available for each town.

In [ ]:
# Display basic information
print("Dataset Shape:")
print(f"  Rows (towns): {len(towns_gdf)}")
print(f"  Columns (attributes): {len(towns_gdf.columns)}")

print(f"\nCoordinate Reference System:")
print(f"  {towns_gdf.crs}")
print(f"  Name: {towns_gdf.crs.name}")

print(f"\nGeometry Type:")
print(f"  {towns_gdf.geometry.type.unique()}")

print(f"\nBounding Box (in meters, Vermont State Plane):")
bounds = towns_gdf.total_bounds
print(f"  Min X: {bounds[0]:,.2f}")
print(f"  Min Y: {bounds[1]:,.2f}")
print(f"  Max X: {bounds[2]:,.2f}")
print(f"  Max Y: {bounds[3]:,.2f}")

### Exploring the Data

Let's examine what we just downloaded. A GeoDataFrame is like a pandas DataFrame but with a special `geometry` column that stores shapes (polygons in this case).

In [ ]:
def fetch_town_boundaries(use_cache: bool = True) -> gpd.GeoDataFrame:
    """
    Fetch Vermont town boundaries from VCGI OpenData portal.
    
    Parameters
    ----------
    use_cache : bool, default True
        If True, load from local cache if available. Otherwise, fetch from API.
    
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing Vermont town boundaries with attributes
    
    Notes
    -----
    Data is cached to data/towns.geojson after first download.
    """
    # Check if cached file exists and use_cache is True
    if use_cache and TOWNS_CACHE.exists():
        print(f"📁 Loading towns from cache: {TOWNS_CACHE}")
        gdf = gpd.read_file(TOWNS_CACHE)
        print(f"✓ Loaded {len(gdf)} towns from cache")
        return gdf
    
    # Fetch from API
    print(f"🌐 Fetching town boundaries from VCGI...")
    print(f"   URL: {TOWN_BOUNDARIES_URL[:80]}...")
    
    try:
        # Make HTTP GET request
        response = requests.get(TOWN_BOUNDARIES_URL, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        
        # Parse GeoJSON response
        geojson_data = response.json()
        
        # Convert to GeoDataFrame
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
        
        # Set coordinate reference system
        # VCGI data is in Vermont State Plane (EPSG:32145)
        gdf.set_crs(VT_STATE_PLANE, inplace=True)
        
        print(f"✓ Fetched {len(gdf)} town boundaries")
        
        # Save to cache
        print(f"💾 Saving to cache: {TOWNS_CACHE}")
        gdf.to_file(TOWNS_CACHE, driver='GeoJSON')
        print(f"✓ Cache saved successfully")
        
        return gdf
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing data: {e}")
        raise

# Fetch the data
towns_gdf = fetch_town_boundaries()

print(f"\n" + "="*50)
print("TOWN BOUNDARIES LOADED SUCCESSFULLY")
print("="*50)

# Interactive Vermont Geology Explorer
## A Geospatial Python Workshop

---

### Workshop Overview

Welcome! This notebook demonstrates a complete geospatial workflow using real-world Vermont GIS data. You'll learn to:

- **Fetch data from REST APIs**: Query ArcGIS REST endpoints to retrieve GeoJSON data
- **Build interactive visualizations**: Create dynamic maps with user controls
- **Perform spatial operations**: Clip, intersect, and analyze vector geometries
- **Analyze spatial data**: Calculate statistics and create visualizations
- **Optimize workflows**: Implement caching to reduce redundant API calls

### What We're Building

By the end of this notebook, you'll have an interactive tool that allows you to:
1. Select any Vermont town from a dropdown menu
2. View bedrock geology on an interactive map
3. Extract geology data for that specific town
4. Analyze the distribution of geologic units
5. Visualize results with charts and tables

### Learning Objectives

**Data Access**
- Query ArcGIS REST Feature Services and Map Services
- Understand REST API parameter structures
- Implement local caching strategies

**Geospatial Analysis**
- Work with coordinate reference systems (CRS)
- Perform spatial queries (bounding box intersections)
- Clip geometries using spatial overlays
- Calculate areas for polygon features

**Interactive Visualization**
- Create interactive web maps with Python
- Add user interface controls (dropdowns, buttons)
- Implement hover and click interactions
- Layer different data sources effectively

**Python Geospatial Ecosystem**
- Use GeoPandas for vector data manipulation
- Apply Shapely for geometric operations
- Build maps with Folium or ipyleaflet
- Leverage ipywidgets for interactivity

### Data Sources

This workshop uses public data from Vermont state agencies:

**1. Town Boundaries**
- **Source**: Vermont Center for Geographic Information (VCGI)
- **Endpoint**: VCGI OpenData Boundary Service
- **License**: Public domain
- **Format**: GeoJSON via ArcGIS REST API

**2. Bedrock Geology**
- **Source**: Vermont Agency of Natural Resources (ANR)
- **Endpoint**: ANR Geologic Map Service
- **License**: Public domain
- **Format**: GeoJSON via query (vector) and tile service (raster)

**3. Basemap**
- **Source**: National Geographic / Esri
- **Type**: Vector tile layer
- **Purpose**: Provides geographic context

### Prerequisites

**Python Knowledge**
- Basic Python syntax and data structures
- Familiarity with pandas DataFrames helpful but not required

**GIS Concepts**
- Basic understanding of coordinate systems (will be explained)
- Familiarity with vector data (points, lines, polygons)
- No prior GIS software experience required!

### Workflow Overview

```
1. Setup & Imports
   ↓
2. Fetch Town Boundaries → Cache Locally
   ↓
3. Create Town Selector UI
   ↓
4. Display Map with Geology Layer
   ↓
5. Query Geology for Selected Town → Cache Locally
   ↓
6. Clip Geology to Town Boundary
   ↓
7. Add Clipped Layer to Map (Interactive)
   ↓
8. Analyze & Visualize Results
```

---

### About This Notebook

This educational material was developed openly and transparently with AI assistance from Claude Code. The workflow demonstrates real-world geospatial analysis patterns you can adapt for your own projects.

**Repository**: [PyDataVT2025](https://github.com/yourusername/PyDataVT2025)

---

Let's get started!

## Environment Setup

First, let's import the required libraries. If you don't have these installed, see the `requirements.txt` file in the repository.

### Required Libraries:

- **`geopandas`**: GeoPandas extends pandas to work with geospatial data. It combines the capabilities of pandas with geometric operations from shapely.
- **`shapely`**: Library for geometric operations (automatically installed with geopandas)
- **`folium`**: Creates interactive Leaflet maps in Python
- **`requests`**: Simple HTTP library for making API calls
- **`ipywidgets`**: Interactive UI components for Jupyter notebooks
- **`matplotlib`**: Plotting library for charts and visualizations
- **`pathlib`**: Object-oriented filesystem paths (part of standard library)

### Coordinate Reference Systems (CRS)

We'll be working with two coordinate systems:
- **EPSG:32145** - Vermont State Plane (NAD83) - Used by Vermont state data, units in meters
- **EPSG:4326** - WGS84 - Standard for web maps (Google Maps, OpenStreetMap), units in degrees

In [1]:
# Standard library imports
import json
from pathlib import Path
from typing import Optional, Tuple
import warnings

# Third-party imports
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box
import folium
from folium import plugins
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Button, Output, VBox, HBox
from IPython.display import display, HTML

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"\nLibrary Versions:")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Folium: {folium.__version__}")

✓ All libraries imported successfully!

Library Versions:
  GeoPandas: 1.1.1
  Pandas: 2.3.3
  Folium: 0.20.0


### Create Data Directory

We'll cache downloaded data locally to avoid repeated API calls. This is a best practice when working with external data sources:
- Faster execution on subsequent runs
- Reduces load on data providers
- Enables offline work after initial download
- Makes your analysis reproducible

In [2]:
# Create data directory if it doesn't exist
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"✓ Data directory ready: {DATA_DIR.absolute()}")

✓ Data directory ready: c:\Users\Steve\Documents\GitHub\PyDataVT2025\data


### Configuration

Let's define the URLs and parameters we'll use throughout the notebook. Keeping these at the top makes the code easier to maintain and adapt for other regions or data sources.

In [3]:
# API Endpoints
TOWN_BOUNDARIES_URL = (
    "https://services1.arcgis.com/BkFxaEFNwHqX3tAw/arcgis/rest/services/"
    "FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1/FeatureServer/0/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

GEOLOGY_MAPSERVICE_URL = (
    "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/"
    "OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/165"
)

GEOLOGY_QUERY_ENDPOINT = f"{GEOLOGY_MAPSERVICE_URL}/query"

BASEMAP_URL = (
    "https://basemaps.arcgis.com/arcgis/rest/services/"
    "World_Basemap_v2/VectorTileServer"
)

# Coordinate Reference Systems
VT_STATE_PLANE = "EPSG:32145"  # Vermont State Plane NAD83 (meters)
WEB_MERCATOR = "EPSG:3857"      # Web Mercator (for tile services)
WGS84 = "EPSG:4326"             # WGS84 (latitude/longitude)

# File paths for cached data
TOWNS_CACHE = DATA_DIR / "towns.geojson"

print("✓ Configuration complete!")
print(f"\nData Sources:")
print(f"  Towns: VCGI OpenData Portal")
print(f"  Geology: VT Agency of Natural Resources")
print(f"\nCoordinate Systems:")
print(f"  Vermont State Plane: {VT_STATE_PLANE}")
print(f"  WGS84 (Web): {WGS84}")

✓ Configuration complete!

Data Sources:
  Towns: VCGI OpenData Portal
  Geology: VT Agency of Natural Resources

Coordinate Systems:
  Vermont State Plane: EPSG:32145
  WGS84 (Web): EPSG:4326


---

## Ready to Begin!

With our environment configured, we're ready to start fetching and working with geospatial data. In the next section, we'll download Vermont town boundaries and explore the data structure.

### Key Concepts to Remember:

1. **Caching**: We save downloaded data locally to avoid repeated API calls
2. **CRS**: Always be aware of which coordinate system your data uses
3. **REST APIs**: We'll interact with ArcGIS REST services using simple HTTP requests
4. **GeoJSON**: A standard format for encoding geographic data structures

Let's dive in! 🗺️

---

## Step 2: Fetching Vermont Town Boundaries

Now we'll fetch town boundary data from the VCGI OpenData portal. This demonstrates:
- Making HTTP requests to ArcGIS REST endpoints
- Implementing a caching strategy
- Loading GeoJSON into GeoPandas
- Inspecting geospatial data

### Understanding ArcGIS REST Services

ArcGIS REST services are web APIs that provide access to geographic data. The URL we're using includes:
- **Service endpoint**: Points to the specific layer (town boundaries)
- **Query parameters**:
  - `outFields=*` - Return all attribute fields
  - `where=1=1` - SQL clause that returns all features (always true)
  - `f=geojson` - Return format as GeoJSON

### Caching Strategy

We'll check if data exists locally before downloading. This:
- Speeds up subsequent notebook runs
- Reduces server load
- Enables offline work
- Ensures consistent data across runs